# FrustraMPNN: Local Testing Notebook

This notebook is for **local testing** of FrustraMPNN. It uses the same API as the Colab notebook but without Colab-specific features.

## Prerequisites

1. Install the package: `uv sync --all-extras` or `pip install -e .[all]`
2. Have a model checkpoint available
3. Have a test PDB file (e.g., `test_data/1UBQ.pdb`)

## 1. Setup and Imports

In [ ]:
# Check if running in the correct environment
import sys
from pathlib import Path

# Add src to path if running from notebooks directory
repo_root = Path.cwd().parent
if (repo_root / "src").exists():
    sys.path.insert(0, str(repo_root / "src"))
    print(f"Added {repo_root / 'src'} to path")

# Verify imports
try:
    import frustrampnn
    print(f"FrustraMPNN version: {frustrampnn.__version__}")
except ImportError as e:
    print(f"Import error: {e}")
    print("Run: uv sync --all-extras")

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

Added /home/felipe/github/frustraMPNN-2/src to path
FrustraMPNN version: 1.0.0
PyTorch version: 2.9.1+cu128
CUDA available: False


## 2. Configuration

Set paths to your checkpoint and PDB file.

In [ ]:
# Configuration - adjust these paths as needed
from pathlib import Path

# Repository root (assuming running from notebooks/)
REPO_ROOT = Path.cwd().parent

# Model checkpoint path
# Option 1: Use existing checkpoint in inference directory
CHECKPOINT_PATH = REPO_ROOT / "weights" / "local_train_raw_fireprot_nosubtract_seed1_epoch=19_val_frustration_spearman=0.8.ckpt"

# Option 2: Specify custom path
# CHECKPOINT_PATH = Path("/path/to/your/checkpoint.ckpt")

# Test PDB file
PDB_PATH = REPO_ROOT / "test_data" / "1UBQ.pdb"

# Chain to analyze
CHAIN_ID = "A"

# Verify files exist
print(f"Repository root: {REPO_ROOT}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"   Exists: {CHECKPOINT_PATH.exists()}")
print(f"PDB file: {PDB_PATH}")
print(f"   Exists: {PDB_PATH.exists()}")

Repository root: /home/felipe/github/frustraMPNN-2
Checkpoint: /home/felipe/github/frustraMPNN-2/inference/local_train_raw_fireprot_nosubtract_seed1_epoch=19_val_frustration_spearman=0.8.ckpt
   Exists: True
PDB file: /home/felipe/github/frustraMPNN-2/test_data/1UBQ.pdb
   Exists: True


## 3. Load Model

In [5]:
from frustrampnn import FrustraMPNN

print("Loading FrustraMPNN model...")
print(f"   Checkpoint: {CHECKPOINT_PATH.name}")

# Load model
model = FrustraMPNN.from_pretrained(str(CHECKPOINT_PATH))

print(f"\nModel loaded successfully!")
print(f"   {model}")

Loading FrustraMPNN model...
   Checkpoint: local_train_raw_fireprot_nosubtract_seed1_epoch=19_val_frustration_spearman=0.8.ckpt

Model loaded successfully!
   FrustraMPNN(device=cpu)


## 4. Run Prediction

In [ ]:
import time

print(f"Running frustration prediction...")
print(f"   PDB: {PDB_PATH.name}")
print(f"   Chain: {CHAIN_ID}")
print()

# Run prediction
start_time = time.time()

results = model.predict(
    str(PDB_PATH),
    chains=[CHAIN_ID],
    show_progress=True
)

elapsed = time.time() - start_time

# Summary
n_positions = results["position"].nunique()
n_predictions = len(results)

print(f"\nPrediction complete!")
print(f"   Positions analyzed: {n_positions}")
print(f"   Total predictions: {n_predictions} ({n_positions} × 20 amino acids)")
print(f"   Time elapsed: {elapsed:.2f} seconds")
print(f"   Speed: {n_predictions / elapsed:.1f} predictions/second")

# Show sample
print(f"\nSample results:")
print(results.head(10).to_string(index=False))

Running frustration prediction...
   PDB: 1UBQ.pdb
   Chain: A



Chain A:  60%|██████    | 914/1520 [01:50<00:58, 10.44it/s]

## 5. Visualizations

### 5.1 Single-Residue Frustration Plot

In [ ]:
from frustrampnn.visualization import plot_single_residue, plot_single_residue_plotly

# Position to plot (0-indexed)
position_to_plot = 0

# Get available positions
available_positions = sorted(results["position"].unique())
print(f"Available positions: {min(available_positions)} to {max(available_positions)}")

# Matplotlib version
print("\nMatplotlib plot:")
fig = plot_single_residue(
    results,
    position=position_to_plot,
    chain=CHAIN_ID,
)
fig.savefig(f"position_{position_to_plot}_matplotlib.png", dpi=150, bbox_inches="tight")
print(f"   Saved: position_{position_to_plot}_matplotlib.png")

In [ ]:
# Plotly version (interactive)
print("Plotly plot (interactive):")
fig_plotly = plot_single_residue_plotly(
    results,
    position=position_to_plot,
    chain=CHAIN_ID,
    width=900,
    height=500,
)
fig_plotly.show()

### 5.2 Frustration Heatmap

In [ ]:
from frustrampnn.visualization import plot_frustration_heatmap, plot_frustration_heatmap_plotly

# Matplotlib version
print("Matplotlib heatmap:")
fig = plot_frustration_heatmap(
    results,
    chain=CHAIN_ID,
    figsize=(14, 8),
)
fig.savefig("heatmap_matplotlib.png", dpi=150, bbox_inches="tight")
print("   Saved: heatmap_matplotlib.png")

In [ ]:
# Plotly version (interactive)
print("Plotly heatmap (interactive):")
fig_plotly = plot_frustration_heatmap_plotly(
    results,
    chain=CHAIN_ID,
    width=1200,
    height=600,
)
fig_plotly.show()

### 5.3 3D Structure Viewer

In [ ]:
import py3Dmol

def get_frustration_color(value: float) -> str:
    """Convert frustration value to RGB color."""
    # Normalize to 0-1 range (assuming -3 to 3 range)
    normalized = (value + 3) / 6
    normalized = max(0, min(1, normalized))

    # Red (frustrated) to Green (minimally frustrated)
    if normalized < 0.5:
        r = 255
        g = int(255 * (normalized * 2))
        b = 0
    else:
        r = int(255 * (1 - (normalized - 0.5) * 2))
        g = 255
        b = 0

    return f"rgb({r},{g},{b})"

def create_3d_viewer(pdb_path, results_df, chain, color_mode="average"):
    """Create py3Dmol viewer colored by frustration."""
    with open(pdb_path, "r") as f:
        pdb_content = f.read()

    chain_results = results_df[results_df["chain"] == chain]

    if color_mode == "average":
        pos_frustration = chain_results.groupby("position")["frustration_pred"].mean()
    elif color_mode == "wildtype":
        wt_mask = chain_results["mutation"] == chain_results["wildtype"]
        pos_frustration = chain_results[wt_mask].set_index("position")["frustration_pred"]
    else:  # most_frustrated
        pos_frustration = chain_results.groupby("position")["frustration_pred"].min()

    view = py3Dmol.view(width=800, height=600)
    view.addModel(pdb_content, "pdb")

    for pos, frust in pos_frustration.items():
        color = get_frustration_color(frust)
        view.setStyle(
            {"resi": pos + 1, "chain": chain},
            {"cartoon": {"color": color}}
        )

    view.zoomTo()
    view.setBackgroundColor("white")
    return view

# Create viewer
print("3D Structure Viewer:")
viewer = create_3d_viewer(str(PDB_PATH), results, CHAIN_ID, "average")
viewer.show()

## 6. Validation Against frustrapy (Optional)

Note: This is slow! Only run for a few positions.

In [ ]:
# Set to True to run validation
RUN_VALIDATION = False

# Positions to validate (0-indexed)
POSITIONS_TO_VALIDATE = [0, 1, 2]

if RUN_VALIDATION:
    from frustrampnn.validation import compare_with_frustrapy
    import time

    print(f"Running frustrapy validation...")
    print(f"   Positions: {POSITIONS_TO_VALIDATE}")
    print(f"   Chain: {CHAIN_ID}")
    print(f"\nThis may take several minutes...")

    start_time = time.time()

    try:
        comparison = compare_with_frustrapy(
            pdb_path=str(PDB_PATH),
            chain=CHAIN_ID,
            frustrampnn_results=results,
            positions=POSITIONS_TO_VALIDATE,
        )

        elapsed = time.time() - start_time

        print(f"\nValidation complete! ({elapsed:.1f} seconds)")
        print(f"\nCorrelation Metrics:")
        print(f"   Spearman correlation: {comparison.spearman:.3f}")
        print(f"   Pearson correlation:  {comparison.pearson:.3f}")
        print(f"   RMSE:                 {comparison.rmse:.3f}")
        print(f"   MAE:                  {comparison.mae:.3f}")

        # Show comparison plot
        fig = comparison.plot()
        fig.show()

    except Exception as e:
        print(f"Validation failed: {e}")
else:
    print("Validation skipped. Set RUN_VALIDATION = True to enable.")

## 7. Export Results

In [ ]:
from frustrampnn.visualization import classify_frustration

# Prepare export
pdb_name = PDB_PATH.stem
output_filename = f"{pdb_name}_frustration_predictions.csv"

results_export = results.copy()
results_export["frustration_class"] = results_export["frustration_pred"].apply(classify_frustration)
results_export["position_1indexed"] = results_export["position"] + 1

# Reorder columns
column_order = [
    "pdb", "chain", "position", "position_1indexed",
    "wildtype", "mutation", "frustration_pred", "frustration_class"
]
results_export = results_export[column_order]

# Save
results_export.to_csv(output_filename, index=False)

print(f"Results saved to: {output_filename}")
print(f"   Rows: {len(results_export)}")

# Summary statistics
print(f"\nSummary Statistics:")
print(f"   Mean frustration: {results_export['frustration_pred'].mean():.3f}")
print(f"   Std frustration:  {results_export['frustration_pred'].std():.3f}")
print(f"   Min frustration:  {results_export['frustration_pred'].min():.3f}")
print(f"   Max frustration:  {results_export['frustration_pred'].max():.3f}")

# Class distribution
class_counts = results_export["frustration_class"].value_counts()
print(f"\nFrustration Class Distribution:")
for cls, count in class_counts.items():
    pct = 100 * count / len(results_export)
    print(f"   {cls}: {count} ({pct:.1f}%)")

## 8. Compare with Reference Output (Regression Test)

In [ ]:
import pandas as pd
import numpy as np

# Reference output path
REFERENCE_PATH = REPO_ROOT / "test_data" / "1UBQ_reference_output.csv"

if REFERENCE_PATH.exists():
    print(f"Comparing with reference output...")
    print(f"   Reference: {REFERENCE_PATH.name}")

    # Load reference
    reference = pd.read_csv(REFERENCE_PATH)

    # Merge on position, wildtype, mutation, chain
    merged = results.merge(
        reference,
        on=["position", "wildtype", "mutation", "chain"],
        suffixes=("_new", "_ref")
    )

    if len(merged) == 0:
        print("No matching rows found. Check column names.")
    else:
        # Calculate differences
        diff = np.abs(merged["frustration_pred_new"] - merged["frustration_pred_ref"])

        print(f"\nComparison Results:")
        print(f"   Matched rows: {len(merged)}")
        print(f"   Max difference: {diff.max():.6f}")
        print(f"   Mean difference: {diff.mean():.6f}")
        print(f"   Std difference: {diff.std():.6f}")

        # Check tolerance
        tolerance = 1e-5
        n_exceed = (diff > tolerance).sum()

        if n_exceed == 0:
            print(f"\nPASSED - All predictions within tolerance ({tolerance})")
        else:
            print(f"\nFAILED - {n_exceed} predictions exceed tolerance ({tolerance})")
            # Show worst offenders
            worst = merged.loc[diff.nlargest(5).index]
            print("\nWorst differences:")
            for _, row in worst.iterrows():
                print(f"   pos={row['position']}, wt={row['wildtype']}, mut={row['mutation']}: "
                      f"new={row['frustration_pred_new']:.6f}, ref={row['frustration_pred_ref']:.6f}")
else:
    print(f"Reference file not found: {REFERENCE_PATH}")
    print("   Skipping regression test.")

## 9. Test CLI (Optional)

In [ ]:
import subprocess

# Test CLI help
print("Testing CLI...")
result = subprocess.run(
    ["python", "-m", "frustrampnn.cli", "--help"],
    capture_output=True,
    text=True,
    cwd=str(REPO_ROOT)
)

if result.returncode == 0:
    print("CLI works!")
    print(result.stdout[:500] + "..." if len(result.stdout) > 500 else result.stdout)
else:
    print(f"CLI error: {result.stderr}")

---

## Summary

This notebook tested:
1. Package imports
2. Model loading
3. Frustration prediction
4. Single-residue plots (matplotlib + plotly)
5. Heatmap plots (matplotlib + plotly)
6. 3D structure viewer
7. Validation against frustrapy (optional)
8. Results export
9. Regression test against reference
10. CLI functionality